# 🛗 Elevator Predictive Maintenance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/elevator-maintenance/training/notebook.ipynb)

From the **last 60 minutes** of 11 elevator sensors (one reading per minute), predict whether the elevator
will be in a **degradation/failure state 10 minutes later**. We compare a **RandomForest on 63 engineered
window features** (rolling stats, slopes, FFT of vibration, time of day) with a **Keras 1D-CNN** and a
**Keras LSTM** on the raw windows, choose on validation F1, and serve the winner in the
[Space](https://huggingface.co/spaces/shalev396/elevator-maintenance) from the [model repo](https://huggingface.co/shalev396/elevator-maintenance).

> ⚠️ **The data is SYNTHETIC.** The Kaggle set usually linked to this brief
> (`shivamb/elevator-predictive-maintenance-dataset`) has no failure label, so `src/data_setup.py` generates
> a seeded, spec-matched month instead (44,640 minutely rows, 5 degradation episodes). The scores below measure
> how well each model learns that generator, not how it would do on a real elevator.

All logic lives in `src/` (and `../model/model.py` for features + inference); this notebook calls it step by step.

1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "elevator-maintenance"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has; nothing platform-specific is installed or special-cased.
import tensorflow as tf
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("device:", device)

In [ ]:
import time

import matplotlib.pyplot as plt  # activates inline figures
import pandas as pd
from IPython.display import display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config

import model as M  # ../model/model.py, importable once `src` is imported

print(utils.lib_versions("sklearn", "tensorflow", "keras", "numpy", "pandas", "scipy"))

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` runs the same pipeline on every 20th window
with 50 trees, tiny nets and 1 epoch (well under a minute on CPU) and exports to `outputs/smoke/` instead of
`../model/` and `../space/examples/`.

In [ ]:
# "0" = full training (~1-2 min on a desktop CPU), "1" = smoke run. setdefault keeps a value exported by the
# caller (e.g. `SMOKE_TEST=1 jupyter nbconvert --execute ...`); restart the kernel after changing it.
os.environ.setdefault("SMOKE_TEST", "0")
PUSH_TO_HUB = False  # upload the exported model repo to the Hub at the end (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "tensorflow")
t0 = time.perf_counter()
cfg

## 3. Data
**Synthetic month** (seeded generator in `src/data_setup.py`, cached to `data/elevator.csv`): a daily usage
profile (rush hours, quiet nights and weekends) drives all 11 sensors plus noise. Five **degradation episodes**
add 6-14 h of drift in vibration, motor current, temperature, noise and power, faster oil loss, then a
45-120 min failure and a maintenance reset. `Status = 1` during degradation + failure (8.5 % of minutes).
Anchors are placed so that train, validation **and** test each hold at least one full episode.

In [ ]:
df = data_setup.load_elevator(cfg)
display(pd.DataFrame(data_setup.episodes(df), columns=["episode start", "episode end"]))
print(f"{len(df):,} rows x {len(M.SENSORS)} sensors, Status=1 on {df['Status'].mean():.1%} of minutes")
df.head()

In [ ]:
display(export.plot_month(df, cfg))
display(export.plot_daily_profile(df))
display(export.plot_lowpass(df))  # a Butterworth low-pass (scipy.signal) shows the drift under the noise

**Windows.** The rows are split by time 70/15/15. Inside each segment, a window covers minutes `[t-59, t]` and
its label is `Status` at `t+10`; both must lie in the same segment, so **nothing straddles a split boundary**.
Stride 5 minutes. The forest sees 63 features per window from `model.window_features` (the exact code the
Space runs): mean/std/min/max/slope of each sensor, the top-5 FFT magnitudes of `vibration_rms` + its dominant
bin, hour and day of week. The Keras nets see the raw window, standardised with a scaler fit on train rows.

In [ ]:
windows = data_setup.build_windows(df, cfg)
feats = data_setup.features(windows, cfg)
display(windows.summary())
print("raw windows:", windows.raw["train"].shape, "| features:", feats["train"].shape)
pd.DataFrame(feats["train"][:3], columns=M.feature_names()).T.head(12)

## 4. Load model
Three experiments. The forest's constructor comes from `../model/model.py` (the deployed model); the two
Keras nets are comparison models built in `src/model_builder.py`.

In [ ]:
models = model_builder.build_all(cfg)
for name in models:
    print(f"{name:13s} {model_builder.describe(name, cfg)}")
models[model_builder.RF]

In [ ]:
for name in (model_builder.CNN, model_builder.LSTM):
    models[name].summary()
    print(f"{name}: {model_builder.params(name, models[name]):,} parameters")

## 5. Training
The forest is fit on the train features (balanced class weights). The nets use Adam, binary cross-entropy,
class weights (failure windows are ~7 % of train) and early stopping on validation PR-AUC.

In [ ]:
train_time = {model_builder.RF: engine.fit_forest(models[model_builder.RF], feats, windows)}
print(f"RandomForest fit in {train_time[model_builder.RF]:.1f} s")
histories = {}
for name in (model_builder.CNN, model_builder.LSTM):
    print(f"--- {name}")
    histories[name], train_time[name] = engine.fit_keras(models[name], windows, cfg)

In [ ]:
display(export.plot_histories(histories))

## 6. Evaluation
For every experiment the decision threshold is set to maximise F1 on **validation**; the model is chosen by
validation F1; the **test** segment (the last 4.65 days, one unseen episode) is scored once at that threshold.

In [ ]:
evals = {}
for name, net in models.items():
    prob = engine.probabilities(name, net, windows, feats)
    evals[name] = engine.evaluate(name, prob, windows, cfg, train_time[name], model_builder.params(name, net))
best = engine.select_best(evals)
print("selected on validation:", best)
engine.comparison_table(evals).round(4)

In [ ]:
chosen = evals[best]
cm = engine.confusion(windows.y["test"], chosen.prob["test"], chosen.threshold)
display(export.plot_comparison(evals, best))
display(export.plot_pr_curves(evals, best, windows.y["test"]))
display(export.plot_confusion(cm, best, chosen.threshold))
display(export.plot_timeline(windows, chosen))
display(export.plot_importance(models[model_builder.RF]))

## 7. Inference
Save the forest + its `config.json` to `cfg.model_dir`, cut three 60-minute example windows from the **test**
segment (healthy / halfway into a degradation / during the failure) and score them with
`model.load(dir).predict(csv)`, exactly what the Space and the endpoint run.

In [ ]:
export.save_model(cfg, models[model_builder.RF], chosen.threshold)
example_files = data_setup.export_examples(df, cfg)
predictor = M.load(cfg.model_dir, device)
example_probs = {}
for name, path in example_files.items():
    pred = predictor.predict(path)
    example_probs[name] = pred["failure"]
    verdict = "MAINTENANCE" if pred["failure"] >= predictor.threshold else "healthy"
    print(f"{name:10s} P(failure) = {pred['failure']:.3f} -> {verdict} (threshold {predictor.threshold:.2f})")
display(export.plot_window(pd.read_csv(example_files["degrading"]), "degrading.csv: the 60 minutes the model sees"))

## 8. Export
Writes `metrics.json`, the card plots in `assets/` and the refreshed model card to `cfg.model_dir` (after
checking that `model.py` reproduces the in-memory test predictions). `cfg.model_dir` is then a complete
Hugging Face model repo. Set `PUSH_TO_HUB = True` after a full run to upload it; the token comes from
`hf auth login` or an `HF_TOKEN` secret / environment variable. Smoke runs are never uploaded.

In [ ]:
metrics = export.export(cfg, evals=evals, best=best, windows=windows, forest=models[model_builder.RF],
                        histories=histories, device=device, total_time_s=time.perf_counter() - t0,
                        examples=example_probs)
print(metrics["primary_metric"], "| threshold", round(metrics["threshold"], 4), "|", metrics["confusion_matrix"])
for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.relative_to(cfg.model_dir).as_posix():<32} {path.stat().st_size / 1e3:>9.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))